In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
TARGET_COL = "readmitted"

df = pd.read_csv("../data/processed/processed.csv").copy()

patient_outcome = (
    df.groupby("patient_nbr")[TARGET_COL]
      .max()
      .rename("patient_has_pos")
      .reset_index()
)

train_pat, temp_pat = train_test_split(
    patient_outcome,
    test_size=0.30,
    random_state=42,
    stratify=patient_outcome["patient_has_pos"]
)

valid_pat, test_pat = train_test_split(
    temp_pat,
    test_size=0.50,
    random_state=42,
    stratify=temp_pat["patient_has_pos"]
)

train_ids = set(train_pat["patient_nbr"])
valid_ids = set(valid_pat["patient_nbr"])
test_ids  = set(test_pat["patient_nbr"])

print("Patient overlap counts (train∩valid, train∩test, valid∩test):",
      len(train_ids & valid_ids), len(train_ids & test_ids), len(valid_ids & test_ids))

train_df = df[df["patient_nbr"].isin(train_ids)].copy()
valid_df = df[df["patient_nbr"].isin(valid_ids)].copy()
test_df  = df[df["patient_nbr"].isin(test_ids)].copy()

def ratio(x): 
    return x[TARGET_COL].mean()

print(f"Train size: {len(train_df):,} | positives: {train_df[TARGET_COL].sum():,} | ratio: {ratio(train_df):.4f}")
print(f"Valid size: {len(valid_df):,} | positives: {valid_df[TARGET_COL].sum():,} | ratio: {ratio(valid_df):.4f}")
print(f"Test  size: {len(test_df):,} | positives: {test_df[TARGET_COL].sum():,} | ratio: {ratio(test_df):.4f}")

cols_to_drop = [c for c in ["patient_nbr", "encounter_id"] if c in df.columns]
train_df_nopid = train_df.drop(columns=cols_to_drop)
valid_df_nopid = valid_df.drop(columns=cols_to_drop)
test_df_nopid  = test_df.drop(columns=cols_to_drop)

Patient overlap counts (train∩valid, train∩test, valid∩test): 0 0 0
Train size: 67,606 | positives: 7,351 | ratio: 0.1087
Valid size: 14,460 | positives: 1,562 | ratio: 0.1080
Test  size: 14,483 | positives: 1,566 | ratio: 0.1081


In [3]:
train_df_nopid.to_csv("../data/processed/training.csv", index=False)
valid_df_nopid.to_csv("../data/processed/validation.csv", index=False)
test_df_nopid.to_csv("../data/processed/testing.csv", index=False)